In [18]:
#pip install pymongo

In [19]:
import os
from pymongo import MongoClient
from pymongo.server_api import ServerApi
import pandas as pd
from pymongo.errors import BulkWriteError, ConnectionFailure

In [20]:
uri = os.environ.get("MONGODB_URI")
if not uri:
    raise RuntimeError("Set the MONGODB_URI environment variable before running this cell.")

client = MongoClient(uri, server_api=ServerApi("1"), serverSelectionTimeoutMS=10000)
try:
    client.admin.command("ping")
    print("Connected successfully")
    client.close()

except Exception as e:
    raise Exception(
        "The following error occurred: ", e)

Connected successfully


In [21]:
#Load the dataset
df = pd.read_csv("pokemon_data.csv")
print(df.head())

                     abilities  against_bug  against_dark  against_dragon  \
0  ['Overgrow', 'Chlorophyll']          1.0           1.0             1.0   
1  ['Overgrow', 'Chlorophyll']          1.0           1.0             1.0   
2  ['Overgrow', 'Chlorophyll']          1.0           1.0             1.0   
3     ['Blaze', 'Solar Power']          0.5           1.0             1.0   
4     ['Blaze', 'Solar Power']          0.5           1.0             1.0   

   against_electric  against_fairy  against_fight  against_fire  \
0               0.5            0.5            0.5           2.0   
1               0.5            0.5            0.5           2.0   
2               0.5            0.5            0.5           2.0   
3               1.0            0.5            1.0           0.5   
4               1.0            0.5            1.0           0.5   

   against_flying  against_ghost  ...  percentage_male  pokedex_number  \
0             2.0            1.0  ...             88.1      

In [22]:
# Replace pandas NaN values with None for MongoDB compatibility
df = df.where(pd.notnull(df), None)

In [23]:
uri = os.environ.get("MONGODB_URI")
if not uri:
    raise RuntimeError("Set the MONGODB_URI environment variable before running this cell.")

client = MongoClient(uri, server_api=ServerApi("1"), serverSelectionTimeoutMS=10000)
try:
    client.admin.command("ping")
    print("Connected successfully")

    #create database and collection
    database = client["bde"]
    collection = database["pokeman"]

    #convert the dataframe into a list of dictionaries
    records = df.to_dict(orient="records")

    #inset records into MongoDB
    if records:
        result = collection.insert_many(records)
        print(f"{len(result.inserted_ids)} records inserted successfully.")
    else:
        print("The CSV file does not contain any records.")   


except ConnectionFailure as error:
    print(f"Could not connect to MongoDB: {error}")

except BulkWriteError as error:
    print("Some records could not be inserted.")
    print(error.details)
    
except Exception as e:
    raise Exception(
        "The following error occurred: ", e)

#finally:
#    if "client" in locals():
#        client.close()
#        print("MongoDB connection closed.")

Connected successfully
801 records inserted successfully.


In [24]:
#inserting a new doc
new_document= {
    "abilities":"Big Data",
    "type1" :"air",
    "type2" :"ice"
}
result = collection.insert_one(new_document)
print("Document inserted successfully.")
print("Inserted ID:", result.inserted_id)

Document inserted successfully.
Inserted ID: 6a5c69dcdd9a24d6fa8e4aa5


In [25]:
#find one record
result = collection.find_one({"abilities": "Big Data"})
print(result)

{'_id': ObjectId('6a5c69addd9a24d6fa8e4781'), 'abilities': 'Big Data', 'type1': 'air', 'type2': 'ice', 'attack': 55, 'course': 'Big Data Engineering'}


In [26]:
#find docs with a condition
for doc in collection.find({"type1": "grass"}):
    print(doc)

{'_id': ObjectId('6a5c69a8dd9a24d6fa8e4461'), 'abilities': "['Overgrow', 'Chlorophyll']", 'against_bug': 1.0, 'against_dark': 1.0, 'against_dragon': 1.0, 'against_electric': 0.5, 'against_fairy': 0.5, 'against_fight': 0.5, 'against_fire': 2.0, 'against_flying': 2.0, 'against_ghost': 1.0, 'against_grass': 0.25, 'against_ground': 1.0, 'against_ice': 2.0, 'against_normal': 1.0, 'against_poison': 1.0, 'against_psychic': 2.0, 'against_rock': 1.0, 'against_steel': 1.0, 'against_water': 0.5, 'attack': 62, 'base_egg_steps': 5120, 'base_happiness': 70, 'base_total': 405, 'capture_rate': '45', 'classfication': 'Seed Pokémon', 'defense': 63, 'experience_growth': 1059860, 'height_m': 1.0, 'hp': 60, 'japanese_name': 'Fushigisouフシギソウ', 'name': 'Ivysaur', 'percentage_male': 88.1, 'pokedex_number': 2, 'sp_attack': 80, 'sp_defense': 80, 'speed': 60, 'type1': 'grass', 'type2': 'poison', 'weight_kg': 13.0, 'generation': 1, 'is_legendary': 0, 'course': 'Big Data Engineering'}
{'_id': ObjectId('6a5c69a8dd9

In [27]:
#find docs with a condition -but limit the result set
for doc in collection.find({"type1": "grass"}).limit(2):
    print(doc)

{'_id': ObjectId('6a5c69a8dd9a24d6fa8e4461'), 'abilities': "['Overgrow', 'Chlorophyll']", 'against_bug': 1.0, 'against_dark': 1.0, 'against_dragon': 1.0, 'against_electric': 0.5, 'against_fairy': 0.5, 'against_fight': 0.5, 'against_fire': 2.0, 'against_flying': 2.0, 'against_ghost': 1.0, 'against_grass': 0.25, 'against_ground': 1.0, 'against_ice': 2.0, 'against_normal': 1.0, 'against_poison': 1.0, 'against_psychic': 2.0, 'against_rock': 1.0, 'against_steel': 1.0, 'against_water': 0.5, 'attack': 62, 'base_egg_steps': 5120, 'base_happiness': 70, 'base_total': 405, 'capture_rate': '45', 'classfication': 'Seed Pokémon', 'defense': 63, 'experience_growth': 1059860, 'height_m': 1.0, 'hp': 60, 'japanese_name': 'Fushigisouフシギソウ', 'name': 'Ivysaur', 'percentage_male': 88.1, 'pokedex_number': 2, 'sp_attack': 80, 'sp_defense': 80, 'speed': 60, 'type1': 'grass', 'type2': 'poison', 'weight_kg': 13.0, 'generation': 1, 'is_legendary': 0, 'course': 'Big Data Engineering'}
{'_id': ObjectId('6a5c69a8dd9

In [28]:
#projection
projection = {
    "_id":0,
    "abilities":1,
    "classfication":1
}
for doc in collection.find({}, projection):
    print(doc)

{'abilities': "['Overgrow', 'Chlorophyll']", 'classfication': 'Seed Pokémon'}
{'abilities': "['Overgrow', 'Chlorophyll']", 'classfication': 'Seed Pokémon'}
{'abilities': "['Blaze', 'Solar Power']", 'classfication': 'Lizard Pokémon'}
{'abilities': "['Blaze', 'Solar Power']", 'classfication': 'Flame Pokémon'}
{'abilities': "['Blaze', 'Solar Power']", 'classfication': 'Flame Pokémon'}
{'abilities': "['Torrent', 'Rain Dish']", 'classfication': 'Tiny Turtle Pokémon'}
{'abilities': "['Torrent', 'Rain Dish']", 'classfication': 'Turtle Pokémon'}
{'abilities': "['Torrent', 'Rain Dish']", 'classfication': 'Shellfish Pokémon'}
{'abilities': "['Shield Dust', 'Run Away']", 'classfication': 'Worm Pokémon'}
{'abilities': "['Shed Skin']", 'classfication': 'Cocoon Pokémon'}
{'abilities': "['Compoundeyes', 'Tinted Lens']", 'classfication': 'Butterfly Pokémon'}
{'abilities': "['Shield Dust', 'Run Away']", 'classfication': 'Hairy Pokémon'}
{'abilities': "['Shed Skin']", 'classfication': 'Cocoon Pokémon'}


In [29]:
#update one doc
result = collection.update_one(
    {"name": "Bulbasaur"},
    {"$set": {"attack": 55}}
)
print("Modified:", result.modified_count)

Modified: 1


In [30]:
#update many docs
result = collection.update_many(
    {"abilities": "Big Data"},
    {"$set": {"attack": 55}}
)
print("Modified:", result.modified_count)

Modified: 1


In [31]:
#Add a new field
result = collection.update_many(
    {},
    {"$set": {"course": "Big Data Engineering"}}
)

print("Modified:", result.modified_count)

Modified: 802


In [32]:
#delete doc
result = collection.delete_one(
    {"name": "Bulbasaur"}
)

print("Deleted:", result.deleted_count)

Deleted: 1


In [16]:
#count docs
count = collection.count_documents({"type1": "grass"})
print("Total Grass Pokémon:", count)

Total Grass Pokémon: 77


In [17]:
if "client" in locals():
    client.close()
    print("MongoDB connection closed.")

MongoDB connection closed.
